# Feature Selection

## Filter Based Technique

In [60]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold, f_classif,SelectKBest

In [61]:
df = pd.read_csv('../datasets/HumanActivitySmartphones/train.csv')

In [62]:
df.head()

,tBodyAcc-mean()-X,tBodyAcc-mean()-Y,tBodyAcc-mean()-Z,tBodyAcc-std()-X,tBodyAcc-std()-Y,tBodyAcc-std()-Z,tBodyAcc-mad()-X,tBodyAcc-mad()-Y,tBodyAcc-mad()-Z,tBodyAcc-max()-X,...,fBodyBodyGyroJerkMag-kurtosis(),"angle(tBodyAccMean,gravity)","angle(tBodyAccJerkMean),gravityMean)","angle(tBodyGyroMean,gravityMean)","angle(tBodyGyroJerkMean,gravityMean)","angle(X,gravityMean)","angle(Y,gravityMean)","angle(Z,gravityMean)",subject,Activity
0,0.288585,-0.020294,-0.132905,-0.995279,-0.983111,-0.913526,-0.995112,-0.983185,-0.923527,-0.934724,...,-0.710304,-0.112754,0.030400,-0.464761,-0.018446,-0.841247,0.179941,-0.058627,1,STANDING
1,0.278419,-0.016411,-0.123520,-0.998245,-0.975300,-0.960322,-0.998807,-0.974914,-0.957686,-0.943068,...,-0.861499,0.053477,-0.007435,-0.732626,0.703511,-0.844788,0.180289,-0.054317,1,STANDING
2,0.279653,-0.019467,-0.113462,-0.995380,-0.967187,-0.978944,-0.996520,-0.963668,-0.977469,-0.938692,...,-0.760104,-0.118559,0.177899,0.100699,0.808529,-0.848933,0.180637,-0.049118,1,STANDING
3,0.279174,-0.026201,-0.123283,-0.996091,-0.983403,-0.990675,-0.997099,-0.982750,-0.989302,-0.938692,...,-0.482845,-0.036788,-0.012892,0.640011,-0.485366,-0.848649,0.181935,-0.047663,1,STANDING
4,0.276629,-0.016570,-0.115362,-0.998139,-0.980817,-0.990482,-0.998321,-0.979672,-0.990441,-0.942469,...,-0.699205,0.123320,0.122542,0.693578,-0.615971,-0.847865,0.185151,-0.043892,1,STANDING


In [63]:
df.shape

(7352, 563)

In [64]:
df['Activity'].value_counts()

Activity
LAYING                1407
STANDING              1374
SITTING               1286
WALKING               1226
WALKING_UPSTAIRS      1073
WALKING_DOWNSTAIRS     986
Name: count, dtype: int64

In [65]:
X = df.drop('Activity', axis=1)
Y = df['Activity']

Y = LabelEncoder().fit_transform(Y)

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

In [66]:
# applying logistic Regression without doing feature selection
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)

In [67]:
accuracy_score(y_test, y_pred)

0.982324949014276

### 1. Removing Duplicate Columns

In [68]:
x_train.shape

(5881, 562)

In [69]:
def get_duplicate_columns(df):
    
    duplicate_columns = {}
    seen_columns = {}

    for column in df.columns:
        current_column = df[column]

        # Convert column data to bytes
        try:
            current_column_hash = current_column.values.tobytes()
        except AttributeError:
            current_column_hash = current_column.to_string().encode()
        if current_column_hash in seen_columns:
            if seen_columns[current_column_hash] in duplicate_columns:
                duplicate_columns[seen_columns[current_column_hash]].append(column)
            else:
                duplicate_columns[seen_columns[current_column_hash]] = [column]
        else:
            seen_columns[current_column_hash] = column
        
    return duplicate_columns

In [70]:
dupli_col = get_duplicate_columns(x_train)

In [71]:
dupli_col.values()

dict_values([['tBodyAccMag-sma()', 'tGravityAccMag-mean()', 'tGravityAccMag-sma()'], ['tGravityAccMag-std()'], ['tGravityAccMag-mad()'], ['tGravityAccMag-max()'], ['tGravityAccMag-min()'], ['tGravityAccMag-energy()'], ['tGravityAccMag-iqr()'], ['tGravityAccMag-entropy()'], ['tGravityAccMag-arCoeff()1'], ['tGravityAccMag-arCoeff()2'], ['tGravityAccMag-arCoeff()3'], ['tGravityAccMag-arCoeff()4'], ['tBodyAccJerkMag-sma()'], ['tBodyGyroMag-sma()'], ['tBodyGyroJerkMag-sma()'], ['fBodyAccMag-sma()'], ['fBodyBodyAccJerkMag-sma()'], ['fBodyBodyGyroMag-sma()'], ['fBodyBodyGyroJerkMag-sma()']])

In [72]:
for i in dupli_col.values():
    x_train.drop(columns=i, inplace=True)
    x_test.drop(columns=i, inplace=False)

In [73]:
x_train.shape

(5881, 541)

### 2. Varience Threshold

In [74]:
vt = VarianceThreshold(0.05)
vt.fit(x_train)

,"threshold threshold: float, default=0Features with a training-set variance lower than this threshold willbe removed. The default is to keep all features with non-zero variance,i.e. remove the features that have the same value in all samples.",0.05


In [75]:
sum(vt.get_support())
# below threshold - > False
# above threshold -> True

np.int64(350)

In [76]:
columns = x_train.columns[vt.get_support()]
columns

Index(['tBodyAcc-std()-X', 'tBodyAcc-std()-Y', 'tBodyAcc-std()-Z',
       'tBodyAcc-mad()-X', 'tBodyAcc-mad()-Y', 'tBodyAcc-mad()-Z',
       'tBodyAcc-max()-X', 'tBodyAcc-max()-Y', 'tBodyAcc-max()-Z',
       'tBodyAcc-min()-X',
       ...
       'fBodyBodyGyroJerkMag-skewness()', 'fBodyBodyGyroJerkMag-kurtosis()',
       'angle(tBodyAccMean,gravity)', 'angle(tBodyAccJerkMean),gravityMean)',
       'angle(tBodyGyroMean,gravityMean)',
       'angle(tBodyGyroJerkMean,gravityMean)', 'angle(X,gravityMean)',
       'angle(Y,gravityMean)', 'angle(Z,gravityMean)', 'subject'],
      dtype='object', length=350)

In [77]:
x_train = pd.DataFrame(x_train, columns=columns)
x_test = pd.DataFrame(x_test, columns=columns)

### 3. Correlation

In [78]:
corr_mat = x_train.corr()
columns = corr_mat.columns

In [79]:
col_to_remove = []

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        if corr_mat.loc[columns[i], columns[j]] > 0.9:
            col_to_remove.append(columns[j])

col_to_remove = set(col_to_remove)
len(col_to_remove)

221

In [80]:
x_train.drop(columns = col_to_remove, inplace=True)
x_test.drop(columns = col_to_remove, inplace=True)

In [81]:
x_train.shape

(5881, 129)

### 4. Anova

In [82]:
sel = SelectKBest(f_classif, k=100)
sel.fit(x_train, y_train)

,"score_func score_func: callable, default=f_classifFunction taking two arrays X and y, and returning a pair of arrays(scores, pvalues) or a single array with scores.Default is f_classif (see below ""See Also""). The default function onlyworks with classification tasks... versionadded:: 0.18",<function f_c...00126E34256C0>
,"k k: int or ""all"", default=10Number of top features to select.The ""all"" option bypasses selection, for use in a parameter search.",100


In [83]:
x_train.columns[sel.get_support()].shape

(100,)

In [84]:
columns = x_train.columns[sel.get_support()]

In [85]:
x_train = sel.transform(x_train)
x_test = sel.transform(x_test)

In [86]:
x_train = pd.DataFrame(x_train, columns=columns)
x_test = pd.DataFrame(x_test, columns=columns)

In [90]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(x_train, y_train)
y_pred = log_reg.predict(x_test)
accuracy_score(y_test, y_pred)

0.9653297076818491